# Notebook 03 — Train All Models (3 Models × 3 Aug Modes = 9 Experiments)

**Run after Notebooks 01 and 02.**

Experiments:
| Exp | Model | Augmentation |
|-----|-------|-------------|
| 01  | EfficientNet-B0 | No Aug |
| 02  | EfficientNet-B0 | GIN Only |
| 03  | EfficientNet-B0 | Full Aug |
| 04  | ResNet50 | No Aug |
| 05  | ResNet50 | GIN Only |
| 06  | ResNet50 | Full Aug |
| 07  | MedSAM ViT-B | No Aug |
| 08  | MedSAM ViT-B | GIN Only |
| 09  | MedSAM ViT-B | Full Aug |

Each cell runs one experiment. Run sequentially or skip any you don't need.
Results are saved to `experiments/<exp_name>/`.

In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Setup
# ─────────────────────────────────────────────
import sys
import json
import yaml
from pathlib import Path

import pandas as pd
import torch

# ── Project paths ────────────────────────────
PROJECT_ROOT = Path('.').resolve().parent

paths = json.loads(
    (PROJECT_ROOT / 'paths.json').read_text()
)

PREPROCESSED = Path(paths['preprocessed'])
EXPERIMENTS  = Path(paths['experiments'])
MEDSAM_CKPT  = Path(paths['medsam_ckpt'])


SPLITS_DIR   = PROJECT_ROOT / 'splits'
CONFIGS_DIR  = PROJECT_ROOT / 'configs'

# ── Add project root to python path ──────────
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Device setup ─────────────────────────────
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(f'Device : {DEVICE}')

if torch.cuda.is_available():

    print(f'GPU    : {torch.cuda.get_device_name(0)}')

    vram_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / 1e9
    )

    print(f'VRAM   : {vram_gb:.1f} GB')

# ── Load splits ──────────────────────────────
train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df   = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df  = pd.read_csv(SPLITS_DIR / 'test.csv')

print('\nDataset splits:')
print(
    f'Train : {len(train_df)} slices | '
    f'{train_df["patient_id"].nunique()} patients'
)

print(
    f'Val   : {len(val_df)} slices | '
    f'{val_df["patient_id"].nunique()} patients'
)

print(
    f'Test  : {len(test_df)} slices | '
    f'{test_df["patient_id"].nunique()} patients'
)

Device : cuda
GPU    : NVIDIA GeForce RTX 4070
VRAM   : 12.9 GB

Dataset splits:
Train : 2285 slices | 157 patients
Val   : 267 slices | 22 patients
Test  : 523 slices | 42 patients


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Helper: build dataloaders
# ─────────────────────────────────────────────
from datasets.fedbca_dataset import FedBCaDataset, build_dataloader

def make_loaders(model_type, batch_size, num_workers=4,
                 use_aug=True, gin_only=False, resnet_aug=False):
                 
    train_ds = FedBCaDataset(
        train_df, str(PREPROCESSED), mode='train',
        model_type=model_type,
        use_augmentation=use_aug,
        use_gin_only=gin_only,
        use_resnet_aug=resnet_aug,
    )
    val_ds = FedBCaDataset(
        val_df, str(PREPROCESSED), mode='val',
        model_type=model_type, use_augmentation=False,
    )
    tl = build_dataloader(train_ds, batch_size, num_workers, use_sampler=True)
    vl = build_dataloader(val_ds,   batch_size, num_workers, use_sampler=False)
    print(f'  Train batches: {len(tl)} | Val batches: {len(vl)}')
    return tl, vl

print('Dataloader helper ready.')

Dataloader helper ready.


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — Helper: build criterion from config
# ─────────────────────────────────────────────
from utils.losses import JointLoss

def make_criterion(config):
    return JointLoss(
        seg_weight     = config.get('seg_weight', 0.85),
        
        cls_weight     = config.get('cls_weight', 0.15),
        focal_gamma    = config.get('focal_gamma', 2.0),
        mibc_alpha     = config.get('mibc_alpha', 1.4),
        ds_weights     = tuple(config.get('ds_weights', [0.50, 0.25])),
        tversky_weight = config.get('tversky_weight', 0.20),
    )

print('Criterion helper ready.')

Criterion helper ready.


In [ ]:
# ─────────────────────────────────────────────
# EXP 01 — EfficientNet, No Augmentation
 
# ─────────────────────────────────────────────
from models.efficientnet.model import EfficientNetMultiTask
from utils.trainer import train_model


with open(CONFIGS_DIR / 'efficientnet.yaml') as f:
    cfg_eff = yaml.safe_load(f)

tl, vl = make_loaders('cnn', cfg_eff['batch_size'], use_aug=False)
model  = EfficientNetMultiTask(pretrained=True)
crit   = make_criterion(cfg_eff)

r01 = train_model(model, tl, vl, crit, cfg_eff,
                  'exp01_eff_noaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP01: DSC={r01["best_dsc"]:.4f} | AUC={r01["best_auc"]:.4f}')

  Train batches: 285 | Val batches: 34


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)


[EfficientNet-B0-CORRECTED] in_ch=5 | params=4,169,692

[exp01_eff_noaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 180
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/180 | Loss=1.2474 (seg=1.468 cls=0.000) | tDSC=0.2445 AUC=0.4554 comb=0.3394★ | LR=1.50e-05 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 100/180 | Loss=0.6553 (seg=0.770 cls=0.004) | tDSC=0.7699 AUC=0.7054 comb=0.7408 | LR=1.44e-05 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 105/180 | Loss=0.6289 (seg=0.739 cls=0.004) | tDSC=0.7688 AUC=0.6607 comb=0.7202 | LR=1.31e-05 | cls_w=0.150

[EARLY STOP] No improvement for 70 epochs.

[exp01_eff_noaug] Done.
  Best DSC : 0.7551
  Best AUC : 0.7857
  Combined : 0.7689

EXP01: DSC=0.7551 | AUC=0.7857


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_best.pt',

In [5]:
# ─────────────────────────────────────────────
# EXP 02 — EfficientNet, GIN Only
# Expected: DSC 0.57-0.67, AUC 0.78-0.85
# Time: ~28 min
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_eff['batch_size'], use_aug=True, gin_only=True)
model  = EfficientNetMultiTask(pretrained=True)
crit   = make_criterion(cfg_eff)

r02 = train_model(model, tl, vl, crit, cfg_eff,
                  'exp02_eff_ginonly', str(EXPERIMENTS), DEVICE)
print(f'\nEXP02: DSC={r02["best_dsc"]:.4f} | AUC={r02["best_auc"]:.4f}')
print(f'GIN gain (EfficientNet): ΔDSC={r02["best_dsc"]-r01["best_dsc"]:+.4f} | '
      f'ΔAUC={r02["best_auc"]-r01["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)


[EfficientNet-B0-CORRECTED] in_ch=5 | params=4,169,692

[exp02_eff_ginonly] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 180
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/180 | Loss=1.2253 (seg=1.442 cls=0.000) | tDSC=0.2332 AUC=0.5625 comb=0.3814★ | LR=1.50e-05 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 145/180 | Loss=0.5421 (seg=0.637 cls=0.006) | tDSC=0.7899 AUC=0.7232 comb=0.7599 | LR=4.38e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 150/180 | Loss=0.5484 (seg=0.645 cls=0.002) | tDSC=0.7886 AUC=0.7143 comb=0.7552 | LR=3.63e-06 | cls_w=0.150

[EARLY STOP] No improvement for 70 epochs.

[exp02_eff_ginonly] Done.
  Best DSC : 0.7909
  Best AUC : 0.8571
  Combined : 0.8207

EXP02: DSC=0.7909 | AUC=0.8571
GIN gain (EfficientNet): ΔDSC=+0.0358 | ΔAUC=+0.0714


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_best.pt',

In [6]:
# ─────────────────────────────────────────────
# EXP 03 — EfficientNet, Full Augmentation
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_eff['batch_size'], use_aug=True)
model  = EfficientNetMultiTask(pretrained=True)
crit   = make_criterion(cfg_eff)

r03 = train_model(model, tl, vl, crit, cfg_eff,
                  'exp03_eff_fullaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP03: DSC={r03["best_dsc"]:.4f} | AUC={r03["best_auc"]:.4f}')
print(f'Full aug gain (vs no aug): '
      f'ΔDSC={r03["best_dsc"]-r01["best_dsc"]:+.4f} | '
      f'ΔAUC={r03["best_auc"]-r01["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)


[EfficientNet-B0-CORRECTED] in_ch=5 | params=4,169,692

[exp03_eff_fullaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 180
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/180 | Loss=1.1877 (seg=1.397 cls=0.000) | tDSC=0.4615 AUC=0.1429 comb=0.3181★ | LR=1.50e-05 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 145/180 | Loss=0.5813 (seg=0.680 cls=0.022) | tDSC=0.8082 AUC=0.7054 comb=0.7619 | LR=4.38e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 150/180 | Loss=0.5796 (seg=0.681 cls=0.004) | tDSC=0.8065 AUC=0.7411 comb=0.7771 | LR=3.63e-06 | cls_w=0.150

[EARLY STOP] No improvement for 70 epochs.

[exp03_eff_fullaug] Done.
  Best DSC : 0.8088
  Best AUC : 0.7679
  Combined : 0.7904

EXP03: DSC=0.8088 | AUC=0.7679
Full aug gain (vs no aug): ΔDSC=+0.0537 | ΔAUC=-0.0178


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_best.pt',

In [7]:
# ─────────────────────────────────────────────
# EXP 04 — ResNet50, No Augmentation
# ─────────────────────────────────────────────
torch.cuda.empty_cache()
from utils.trainer import train_model
from models.resnet50.model import ResNet50MultiTask

with open(CONFIGS_DIR / 'resnet50.yaml') as f:
    cfg_res = yaml.safe_load(f)

tl, vl = make_loaders('cnn', cfg_res['batch_size'], use_aug=False)
model  = ResNet50MultiTask(pretrained=True, use_mixstyle=cfg_res.get('use_mixstyle', True))
crit   = make_criterion(cfg_res)

r04 = train_model(model, tl, vl, crit, cfg_res,
                  'exp04_res50_noaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP04: DSC={r04["best_dsc"]:.4f} | AUC={r04["best_auc"]:.4f}')

  Train batches: 285 | Val batches: 34
[ResNet50-V5] in_ch=5 | mixstyle=True | params=45,884,872

[exp04_res50_noaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 140
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/140 | Loss=0.9110 (seg=1.072 cls=0.000) | tDSC=0.6416 AUC=0.7679 comb=0.6984★ | LR=2.51e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/140 | Loss=0.7051 (seg=0.830 cls=0.000) | tDSC=0.7333 AUC=0.7500 comb=0.7408★ | LR=5.01e-06 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 100/140 | Loss=0.3173 (seg=0.372 cls=0.006) | tDSC=0.7665 AUC=0.7054 comb=0.7390 | LR=2.88e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 105/140 | Loss=0.3337 (seg=0.391 cls=0.008) | tDSC=0.7658 AUC=0.6786 comb=0.7266 | LR=2.36e-06 | cls_w=0.150

[EARLY STOP] No improvement for 30 epochs.

[exp04_res50_noaug] Done.
  Best DSC : 0.7652
  Best AUC : 0.7589
  Combined : 0.7624

EXP04: DSC=0.7652 | AUC=0.7589


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:523: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(


In [5]:
# ─────────────────────────────────────────────
# EXP 05 — ResNet50, GIN Only
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

# ResNet50 uses ResNet50AugmentationPipeline which includes GIN
# use_resnet_aug=True activates ResNet50-specific GIN pipeline
tl, vl = make_loaders('cnn', cfg_res['batch_size'],
                       use_aug=True, gin_only=True)
model  = ResNet50MultiTask(pretrained=True, use_mixstyle=cfg_res.get('use_mixstyle', True))
crit   = make_criterion(cfg_res)

r05 = train_model(model, tl, vl, crit, cfg_res,
                  'exp05_res50_ginonly', str(EXPERIMENTS), DEVICE)
print(f'\nEXP05: DSC={r05["best_dsc"]:.4f} | AUC={r05["best_auc"]:.4f}')
print(f'GIN gain (ResNet50): ΔDSC={r05["best_dsc"]-r04["best_dsc"]:+.4f} | '
      f'ΔAUC={r05["best_auc"]-r04["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34
[ResNet50-V5] in_ch=5 | mixstyle=True | params=45,884,872

[exp05_res50_ginonly] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 140
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/140 | Loss=0.8177 (seg=0.962 cls=0.000) | tDSC=0.6960 AUC=0.3214 comb=0.5274★ | LR=2.51e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/140 | Loss=0.6386 (seg=0.751 cls=0.000) | tDSC=0.7572 AUC=0.2768 comb=0.5410★ | LR=5.01e-06 | cls_w=0.000


...

Ep 115/140 | Loss=0.3026 (seg=0.355 cls=0.003) | tDSC=0.7634 AUC=0.6518 comb=0.7132 | LR=1.48e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 120/140 | Loss=0.3088 (seg=0.363 cls=0.003) | tDSC=0.7658 AUC=0.6607 comb=0.7185 | LR=1.14e-06 | cls_w=0.150

[EARLY STOP] No improvement for 30 epochs.

[exp05_res50_ginonly] Done.
  Best DSC : 0.7629
  Best AUC : 0.7411
  Combined : 0.7531

EXP05: DSC=0.7629 | AUC=0.7411


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:523: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(


NameError: name 'r04' is not defined

In [6]:
# ─────────────────────────────────────────────
# EXP 06 — ResNet50, Full Augmentation
# ─────────────────────────────────────────────
torch.cuda.empty_cache()

tl, vl = make_loaders('cnn', cfg_res['batch_size'],
                       use_aug=True, resnet_aug=True)
model  = ResNet50MultiTask(pretrained=True, use_mixstyle=cfg_res.get('use_mixstyle', True))
crit   = make_criterion(cfg_res)

r06 = train_model(model, tl, vl, crit, cfg_res,
                  'exp06_res50_fullaug', str(EXPERIMENTS), DEVICE)
print(f'\nEXP06: DSC={r06["best_dsc"]:.4f} | AUC={r06["best_auc"]:.4f}')
print(f'Full aug gain (ResNet50): '
      f'ΔDSC={r06["best_dsc"]-r04["best_dsc"]:+.4f} | '
      f'ΔAUC={r06["best_auc"]-r04["best_auc"]:+.4f}')

  Train batches: 285 | Val batches: 34
[ResNet50-V5] in_ch=5 | mixstyle=True | params=45,884,872

[exp06_res50_fullaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 140
  Eff batch : 24
  Rampup    : cls gradient starts ep 20, full at ep 40, combined scoring from ep 45



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/140 | Loss=0.8351 (seg=0.982 cls=0.000) | tDSC=0.6723 AUC=0.4107 comb=0.5546★ | LR=2.51e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/140 | Loss=0.6847 (seg=0.805 cls=0.000) | tDSC=0.7679 AUC=0.3661 comb=0.5871★ | LR=5.01e-06 | cls_w=0.000


...

Ep 120/140 | Loss=0.4457 (seg=0.523 cls=0.005) | tDSC=0.8246 AUC=0.7857 comb=0.8071 | LR=1.14e-06 | cls_w=0.150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:342: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 125/140 | Loss=0.4563 (seg=0.535 cls=0.008) | tDSC=0.8246 AUC=0.7946 comb=0.8111 | LR=8.62e-07 | cls_w=0.150

[EARLY STOP] No improvement for 30 epochs.

[exp06_res50_fullaug] Done.
  Best DSC : 0.8217
  Best AUC : 0.8482
  Combined : 0.8336

EXP06: DSC=0.8217 | AUC=0.8482


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:523: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(


NameError: name 'r04' is not defined

In [4]:
# ─────────────────────────────────────────────
# EXP 07 — MedSAM+Swin Hybrid, No Augmentation
# ─────────────────────────────────────────────

import torch
import yaml

from utils.trainer import train_model
from models.medsam.model import MedSAMSwinHybrid

torch.cuda.empty_cache()

with open(CONFIGS_DIR / 'medsam.yaml') as f:
    cfg_msam = yaml.safe_load(f)

# Override checkpoint path
cfg_msam['medsam_checkpoint'] = MEDSAM_CKPT

assert Path(MEDSAM_CKPT).exists(), (
    f'MedSAM checkpoint not found: {MEDSAM_CKPT}\n'
    'Download from: https://github.com/bowang-lab/MedSAM'
)

# MedSAM uses 256×256 pipeline
tl, vl = make_loaders(
    'cnn',
    cfg_msam['batch_size'],
    use_aug=False
)

model = MedSAMSwinHybrid(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
)

crit = make_criterion(cfg_msam)

r07 = train_model(
    model,
    tl,
    vl,
    crit,
    cfg_msam,
    'exp07_medsam_noaug',
    str(EXPERIMENTS),
    DEVICE
)

print(
    f'\nEXP07: '
    f'DSC={r07["best_dsc"]:.4f} | '
    f'AUC={r07["best_auc"]:.4f}'
)

  Train batches: 571 | Val batches: 67


c:\Users\Mayank\miniconda3\envs\fedbca_v3\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


[MedSAM] removed rel_pos keys : 25
[MedSAM] missing keys         : 25
[MedSAM] unexpected keys      : 0
[MedSAM+Swin FINAL] total=116,049,974 trainable=87,684,150


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)



[exp07_medsam_noaug] Starting training
  Device    : cuda
  AMP       : True
  Epochs    : 200
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


AssertionError: Input height (256) doesn't match model (224).

In [ ]:
# ─────────────────────────────────────────────
# EXP 08 — MedSAM+Swin Hybrid, GIN Only
# ─────────────────────────────────────────────

import torch

from utils.trainer import train_model
from models.medsam.model import MedSAMSwinHybrid

torch.cuda.empty_cache()

# MedSAM uses 256×256 pipeline
tl, vl = make_loaders(
    'cnn',
    cfg_msam['batch_size'],
    use_aug=True,
    gin_only=True
)

model = MedSAMSwinHybrid(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
)

crit = make_criterion(cfg_msam)

r08 = train_model(
    model,
    tl,
    vl,
    crit,
    cfg_msam,
    'exp08_medsam_ginonly',
    str(EXPERIMENTS),
    DEVICE
)

print(
    f'\nEXP08: '
    f'DSC={r08["best_dsc"]:.4f} | '
    f'AUC={r08["best_auc"]:.4f}'
)

print(
    f'GIN gain (MedSAM): '
    f'ΔDSC={r08["best_dsc"]-r07["best_dsc"]:+.4f} | '
    f'ΔAUC={r08["best_auc"]-r07["best_auc"]:+.4f}'
)

In [11]:
# ─────────────────────────────────────────────
# EXP 09 — MedSAM+Swin Hybrid, Full Augmentation
# ─────────────────────────────────────────────

import torch

from utils.trainer import train_model
from models.medsam.model import MedSAMSwinHybrid

torch.cuda.empty_cache()

# MedSAM uses 256×256 pipeline
tl, vl = make_loaders(
    'cnn',
    cfg_msam['batch_size'],
    use_aug=True
)

model = MedSAMSwinHybrid(
    medsam_checkpoint=MEDSAM_CKPT,
    in_channels=5,
)

crit = make_criterion(cfg_msam)

r09 = train_model(
    model,
    tl,
    vl,
    crit,
    cfg_msam,
    'exp09_medsam_fullaug',
    str(EXPERIMENTS),
    DEVICE
)

print(
    f'\nEXP09: '
    f'DSC={r09["best_dsc"]:.4f} | '
    f'AUC={r09["best_auc"]:.4f}'
)

print(
    f'Full aug gain (MedSAM): '
    f'ΔDSC={r09["best_dsc"]-r07["best_dsc"]:+.4f} | '
    f'ΔAUC={r09["best_auc"]-r07["best_auc"]:+.4f}'
)

BEAT_DSC = r09['best_dsc'] >= 0.841
BEAT_AUC = r09['best_auc'] >= 0.866

print(
    f'\nBeats FedBCa DSC 0.841: '
    f'{"✓ YES" if BEAT_DSC else "✗ no"}'
)

print(
    f'Beats FedBCa AUC 0.866: '
    f'{"✓ YES" if BEAT_AUC else "✗ no"}'
)

NameError: name 'cfg_msam' is not defined

In [4]:
  # ─────────────────────────────────────────────
# EXP 10 — Swin Hybrid, No Augmentation
# Expected: DSC 0.60-0.70, AUC 0.79-0.86
# Time: ~45 min (RTX 4070)
# VRAM: ~7.5 GB
# ─────────────────────────────────────────────

import torch
import yaml

torch.cuda.empty_cache()

from utils.trainer import train_model
from models.swin_hybrid.model import SwinHybridMultiTask

# ── Load config ──────────────────────────────
with open(CONFIGS_DIR / 'swin_hybrid.yaml') as f:
    cfg_swin = yaml.safe_load(f)

# IMPORTANT:
# Swin models use 224×224 preprocessing
# therefore use 'swin' loader, NOT 'cnn'
tl, vl = make_loaders(
    'swin',
    cfg_swin['batch_size'],
    use_aug=False
)

# ── Build model ──────────────────────────────
model = SwinHybridMultiTask(
    in_channels=5,
    use_mixstyle=cfg_swin.get('use_mixstyle', True),
)

# ── Build criterion ──────────────────────────
crit = make_criterion(cfg_swin)

# ── Train ────────────────────────────────────
r10 = train_model(
    model,
    tl,
    vl,
    crit,
    cfg_swin,
    'exp10_swin_noaug',
    str(EXPERIMENTS),
    DEVICE
)

print(
    f'\nEXP10: '
    f'DSC={r10["best_dsc"]:.4f} | '
    f'AUC={r10["best_auc"]:.4f}'
)

  Train batches: 380 | Val batches: 45


[SwinHybrid-V6] in_ch=5 | mixstyle=True | params=16,490,366 | trainable=16,487,166


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)



[exp10_swin_noaug] Starting training
  Device    : cuda
  AMP       : False
  Epochs    : 250
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 200/250 | Loss=0.2561 (seg=0.319 cls=0.006) | tDSC=0.8151 AUC=0.8929 comb=0.8501 | LR=8.89e-07 | cls_w=0.200


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 205/250 | Loss=0.2668 (seg=0.332 cls=0.004) | tDSC=0.8143 AUC=0.8929 comb=0.8496 | LR=7.80e-07 | cls_w=0.200

[EARLY STOP] No improvement for 80 epochs.

[exp10_swin_noaug] Done.
  Best DSC : 0.8209
  Best AUC : 0.8929
  Combined : 0.8533

EXP10: DSC=0.8209 | AUC=0.8929


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_best.pt',

In [6]:
# ─────────────────────────────────────────────
# EXP 11 — Swin Hybrid, GIN Only
# ─────────────────────────────────────────────

import torch

torch.cuda.empty_cache()

tl, vl = make_loaders(
    'swin',   # IMPORTANT: use swin loader (224x224)
    cfg_swin['batch_size'],
    use_aug=True,
    gin_only=True
)

model = SwinHybridMultiTask(
    in_channels=5,
    use_mixstyle=cfg_swin.get('use_mixstyle', True),
)

crit = make_criterion(cfg_swin)

r11 = train_model(
    model,
    tl,
    vl,
    crit,
    cfg_swin,
    'exp11_swin_ginonly',
    str(EXPERIMENTS),
    DEVICE
)

print(
    f'\nEXP11: '
    f'DSC={r11["best_dsc"]:.4f} | '
    f'AUC={r11["best_auc"]:.4f}'
)

print(
    f'GIN gain (Swin): '
    f'ΔDSC={r11["best_dsc"]-r10["best_dsc"]:+.4f} | '
    f'ΔAUC={r11["best_auc"]-r10["best_auc"]:+.4f}'
)

  Train batches: 380 | Val batches: 45
[SwinHybrid-V6] in_ch=5 | mixstyle=True | params=16,490,366 | trainable=16,487,166

[exp11_swin_ginonly] Starting training
  Device    : cuda
  AMP       : False
  Epochs    : 250
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/250 | Loss=0.8139 (seg=1.017 cls=0.000) | tDSC=0.6719 AUC=0.5268 comb=0.6066★ | LR=3.01e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/250 | Loss=0.6356 (seg=0.794 cls=0.000) | tDSC=0.7514 AUC=0.6696 comb=0.7146★ | LR=6.00e-06 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 245/250 | Loss=0.4069 (seg=0.508 cls=0.004) | tDSC=0.8364 AUC=0.8393 comb=0.8377 | LR=3.06e-07 | cls_w=0.200


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 250/250 | Loss=0.4138 (seg=0.516 cls=0.005) | tDSC=0.8361 AUC=0.8482 comb=0.8416 | LR=3.00e-07 | cls_w=0.200

[exp11_swin_ginonly] Done.
  Best DSC : 0.8370
  Best AUC : 0.8571
  Combined : 0.8461

EXP11: DSC=0.8370 | AUC=0.8571
GIN gain (Swin): ΔDSC=+0.0161 | ΔAUC=-0.0358


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_best.pt',

In [8]:
# ─────────────────────────────────────────────
# EXP 12 — Swin Hybrid, Full Augmentation
# ─────────────────────────────────────────────

import torch
from utils.trainer import train_model
from models.swin_hybrid.model import SwinHybridMultiTask

torch.cuda.empty_cache()

# IMPORTANT:
# Swin models require 224×224 inputs
# so use 'swin' loader, NOT 'cnn'

tl, vl = make_loaders(
    'swin',
    cfg_swin['batch_size'],
    use_aug=True
)

model = SwinHybridMultiTask(
    in_channels=5,
    use_mixstyle=cfg_swin.get('use_mixstyle', True),
)

crit = make_criterion(cfg_swin)

r12 = train_model(
    model,
    tl,
    vl,
    crit,
    cfg_swin,
    'exp12_swin_fullaug',
    str(EXPERIMENTS),
    DEVICE
)

print(
    f'\nEXP12: '
    f'DSC={r12["best_dsc"]:.4f} | '
    f'AUC={r12["best_auc"]:.4f}'
)

print(
    f'Full aug gain (Swin): '
    f'ΔDSC={r12["best_dsc"]-r10["best_dsc"]:+.4f} | '
    f'ΔAUC={r12["best_auc"]-r10["best_auc"]:+.4f}'
)

  Train batches: 380 | Val batches: 45
[SwinHybrid-V6] in_ch=5 | mixstyle=True | params=16,490,366 | trainable=16,487,166

[exp12_swin_fullaug] Starting training
  Device    : cuda
  AMP       : False
  Epochs    : 250
  Eff batch : 24
  Rampup    : cls gradient starts ep 10, full at ep 20, combined scoring from ep 25



C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = GradScaler(enabled=use_amp)
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep   5/250 | Loss=0.7845 (seg=0.981 cls=0.000) | tDSC=0.6745 AUC=0.4554 comb=0.5759★ | LR=3.01e-06 | cls_w=0.000


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep  10/250 | Loss=0.6621 (seg=0.828 cls=0.000) | tDSC=0.7440 AUC=0.6339 comb=0.6945★ | LR=6.00e-06 | cls_w=0.000


...

C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 105/250 | Loss=0.4654 (seg=0.579 cls=0.011) | tDSC=0.8444 AUC=0.8214 comb=0.8341 | LR=4.07e-06 | cls_w=0.200


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\metrics.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp):


Ep 110/250 | Loss=0.4451 (seg=0.549 cls=0.029) | tDSC=0.8482 AUC=0.8304 comb=0.8402 | LR=3.89e-06 | cls_w=0.200

[EARLY STOP] No improvement for 80 epochs.

[exp12_swin_fullaug] Done.
  Best DSC : 0.8123
  Best AUC : 0.8750
  Combined : 0.8405

EXP12: DSC=0.8123 | AUC=0.8750
Full aug gain (Swin): ΔDSC=-0.0086 | ΔAUC=-0.0179


C:\Users\Mayank\Desktop\fedbca_medsam_v3\fedbca_medsam\utils\trainer.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(save_dir / 'checkpoint_best.pt',